In [ ]:
import asyncio
import sys
from typing import List, Optional

# 假设我们已经在项目中定义了上述Agent类和相关类型
from pi_agent import Agent, AgentMessage, AgentTool, ThinkingLevel,AgentToolResult,AgentEvent,CustomAgentMessage
from nova_ai import Message, get_model, ImageContent, UserMessage, TextContent
import os
os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"
# ============================================================
# 1. 定义自定义工具
# ============================================================

class WeatherTool(AgentTool):
    """查询天气的工具"""
    
    def __init__(self):
        super().__init__(
            name="get_weather",
            description="获取指定城市的天气信息",
            parameters={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "温度单位"
                    }
                },
                "required": ["city"]
            }
        )
        self.label = "天气查询"
    
    async def execute(self, tool_call_id: str, params: dict, 
                     signal: Optional[asyncio.Event] = None,
                     on_update=None) -> AgentToolResult:
        """执行天气查询"""
        city = params["city"]
        unit = params.get("unit", "celsius")
        
        # 模拟API调用
        await asyncio.sleep(1)
        
        # 模拟流式更新
        if on_update:
            on_update(AgentToolResult(
                content=[{"type": "text", "text": f"正在查询{city}的天气..."}],
                details={"status": "querying"}
            ))
            await asyncio.sleep(0.5)
        
        # 返回结果
        weather_data = {
            "city": city,
            "temperature": 25 if unit == "celsius" else 77,
            "condition": "晴朗",
            "humidity": 60
        }
        
        return AgentToolResult(
            content=[
                TextContent(
                    type = "text", 
                    text = f"{city}天气：{weather_data['condition']}，温度{weather_data['temperature']}°{'C' if unit == 'celsius' else 'F'}"
                )
            ],
            details=weather_data
        )


class CalculatorTool(AgentTool):
    """计算器工具"""
    
    def __init__(self):
        super().__init__(
            name="calculate",
            description="执行数学计算",
            parameters={
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式"
                    }
                },
                "required": ["expression"]
            }
        )
        self.label = "计算器"
    
    async def execute(self, tool_call_id: str, params: dict,
                     signal: Optional[asyncio.Event] = None,
                     on_update=None) -> AgentToolResult:
        """执行计算"""
        expr = params["expression"]
        
        try:
            # 注意：在实际应用中应该使用安全的表达式求值方法
            result = eval(expr)
            return AgentToolResult(
                content=[
                    TextContent(
                        type="text", 
                        text=f"{expr} = {result}"
                    )
                ],
                details={"expression": expr, "result": result}
            )
        except Exception as e:
            return AgentToolResult(
                content=[
                    TextContent(
                        type="text", 
                        text=f"计算错误：{str(e)}"
                    )
                ],
                details={"error": str(e)}
            )


# ============================================================
# 2. 自定义消息转换器
# ============================================================

async def custom_convert_to_llm(messages: List[AgentMessage]) -> List[Message]:
    """自定义消息转换器：处理自定义消息类型"""
    result = []
    for msg in messages:
        if msg.role in ("user", "assistant", "toolResult"):
            result.append(msg)
        elif msg.role == "system_notification":
            # 将系统通知转换为用户消息
            result.append({
                "role": "user",
                "content": [{"type": "text", "text": f"[系统通知] {msg.content}"}]
            })
    return result


# ============================================================
# 3. 主使用示例
# ============================================================

async def main():
    print("=" * 50)
    print("Agent 类使用示例")
    print("=" * 50)
    
    # 3.1 初始化Agent
    print("\n[1] 初始化Agent...")
    agent = Agent(
        initial_state={
            "systemPrompt": "你是一个有帮助的助手，可以使用工具来回答用户的问题。",
            "thinkingLevel": "medium"  # 使用中等思考级别
        },
        steering_mode="one-at-a-time",  # 每次处理一个steering消息
        follow_up_mode="all",            # 一次性处理所有follow-up消息
        max_retry_delay_ms=30000          # 最大重试延迟30秒
    )
    
    # 3.2 设置模型
    print("[2] 配置模型和工具...")
    model = get_model("volcengine", "deepseek-r1-250528")
    agent.set_model(model)
    
    # 3.3 添加工具
    tools = [WeatherTool(), CalculatorTool()]
    agent.set_tools(tools)
    
    # 3.4 注册事件监听器
    def on_event(event: AgentEvent):
        """处理所有Agent事件"""
        event_type = event.type
        
        if event_type == "message_start":
            msg = event.message
            print(f"\n[消息开始] {msg.role}: ...")
        
        # elif event_type == "message_update":
        #     msg = event.message
        #     if msg.role == "assistant":
        #         # 打印流式更新的文本内容
        #         for content in msg.content:
        #             if content.type == "text" and content.text:
        #                 print(content.text, end="", flush=True)
        
        elif event_type == "message_end":
            msg = event.message
            print(msg.content.text)
            print(f"\n[消息完成] {msg.role}")
        
        elif event_type == "tool_execution_start":
            print(f"\n[工具开始] {event.tool_name}({event.args})")
        
        elif event_type == "tool_execution_update":
            print(f"  [工具更新] {event.partialResult}")
        
        elif event_type == "tool_execution_end":
            status = "✓ 成功" if not event.isError else "✗ 失败"
            print(f"[工具结束] {event.tool_name} {status}")
        
        elif event_type == "turn_start":
            print("\n--- 新回合开始 ---")
        
        elif event_type == "turn_end":
            print(f"--- 回合结束 (工具结果数: {len(event.toolResults)}) ---\n")
        
        elif event_type == "agent_start":
            print("\n=== Agent 启动 ===\n")
        
        elif event_type == "agent_end":
            print(f"\n=== Agent 结束 (共 {len(event.messages)} 条消息) ===\n")
    
    agent.subscribe(on_event)
    
    # 3.5 发送普通文本消息
    print("\n[3] 发送第一个问题...")
    await agent.prompt("今天北京天气怎么样？")
    
    # 等待agent完成
    await agent.wait_for_idle()
    
    # 3.6 发送包含图片的消息
    # print("\n[4] 发送带图片的消息...")
    # image_content = ImageContent(
    #     type="image",
    #     url="https://example.com/screenshot.png",
    #     detail="high"
    # )
    # await agent.prompt("这张截图里有什么？", images=[image_content])
    
    # await agent.wait_for_idle()
    
    # 3.7 使用steering中断当前操作
    print("\n[5] 演示steering功能...")
    
    # 先发送一个可能耗时的请求
    prompt_task = asyncio.create_task(
        agent.prompt("帮我计算一下 (1500 * 3.14) / 2.5 + 100 * 50 的结果，然后解释计算过程")
    )
    
    # 稍等片刻，然后发送steering消息
    await asyncio.sleep(2)
    print("\n[用户中断] 发送steering消息...")
    
    steering_msg: AgentMessage = UserMessage(
        role="user",
        content=[
            TextContent(
                type="text", 
                text="等等，先告诉我计算器工具是否可用？"
            )
        ],
        timestamp=asyncio.get_event_loop().time()
    )
        
    agent.steer(steering_msg)
    
    # 等待完成
    await prompt_task
    
    # 3.8 使用follow-up队列
    print("\n[6] 演示follow-up功能...")
    
    await agent.prompt("给我讲个笑话")
    
    # 在agent运行时添加follow-up消息
    follow_up1: AgentMessage = UserMessage(
        role="user",
        content=[
            TextContent(
                type="text", 
                text="再来一个科技相关的笑话"
            )
        ],
        timestamp=asyncio.get_event_loop().time()
    )
    follow_up2: AgentMessage = UserMessage(
        role="user",
        content=[
            TextContent(
                type="text", 
                text="最后来一个程序员笑话"
            )
        ],
        timestamp=asyncio.get_event_loop().time()
    )
    
    agent.follow_up(follow_up1)
    agent.follow_up(follow_up2)
    
    print(f"队列状态: 有follow-up消息? {agent.has_queued_messages()}")
    
    # 继续处理队列
    await agent.continue_()
    
    # 3.9 查看历史消息
    print("\n[7] 查看对话历史...")
    for i, msg in enumerate(agent.state.messages, 1):
        role = msg.role
        if role == "user":
            print(f"{i}. 用户: {msg.content}")
        elif role == "assistant":
            # 提取文本内容
            texts = [c.text for c in msg.content if c.type == "text"]
            print(f"{i}. 助手: {texts[0] if texts else '...'}")
        elif role == "toolResult":
            print(f"{i}. 工具结果 [{msg.tool_name}]: {msg.content}")
    
    # 3.10 重置agent
    print("\n[8] 重置agent...")
    agent.reset()
    print(f"消息数: {len(agent.state.messages)}")
    print(f"队列中有消息? {agent.has_queued_messages()}")


# ============================================================
# 4. 高级用法：自定义转换器和上下文变换
# ============================================================

async def advanced_example():
    """展示高级配置选项"""
    print("\n" + "=" * 50)
    print("高级配置示例")
    print("=" * 50)
    
    async def transform_context(messages: List[AgentMessage], 
                                signal: Optional[asyncio.Event] = None) -> List[AgentMessage]:
        """自定义上下文转换：限制上下文长度"""
        # 模拟token计数和截断
        if len(messages) > 10:
            print("[上下文转换] 截断历史消息到最近10条")
            return messages[-10:]
        return messages
    
    async def dynamic_api_key(provider: str) -> Optional[str]:
        """动态获取API密钥"""
        print(f"[API密钥] 为提供商 {provider} 获取密钥")
        # 这里可以从环境变量、密钥管理服务等获取
        return "your-api-key-here"
    
    agent = Agent(
        transform_context=transform_context,
        get_api_key=dynamic_api_key,
        thinking_budgets={
            "low": 1000,
            "medium": 4000,
            "high": 8000
        }
    )
    
    # 使用自定义消息类型
    class NotificationMessage(CustomAgentMessage):
        role = "system_notification"
        content: str
    
    agent.set_system_prompt("你是一个专业的助手")
    agent.set_thinking_level("high")
    
    # 发送消息
    await agent.prompt("解释量子计算的基本原理")
    await agent.wait_for_idle()


# ============================================================
# 5. 运行示例
# ============================================================

if __name__ == "__main__":
    try:
        # 运行主示例
        await main()
        
        # 运行高级示例
        # asyncio.run(advanced_example())
        
    except KeyboardInterrupt:
        print("\n\n用户中断执行")
        sys.exit(0)
    except Exception as e:
        print(f"\n错误: {e}")
        sys.exit(1)

Agent 类使用示例

[1] 初始化Agent...
[2] 配置模型和工具...

[3] 发送第一个问题...

=== Agent 启动 ===


--- 新回合开始 ---

[消息开始] user: ...

[消息完成] user

[消息开始] assistant: ...
查询查询北京的查询北京的天气查询北京的天气信息查询北京的天气信息，查询北京的天气信息，请查询北京的天气信息，请稍查询北京的天气信息，请稍等查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...查询北京的天气信息，请稍等...
[消息完成] assistant

[工具开始] get_weather({'city': '北京', 'unit': 'celsius'})

[消息开始] toolResult: ...

[消息完成] toolResult

--- 新回合开始 ---

[消息开始] assistant: ...
今天今天北京的今天北京的天气今天北京的天气晴朗今天北京的天气晴朗，今天北京的天气晴朗，温度为今天北京的天气晴朗，温度为25今天北京的天气晴朗，温度为25°今天北京的天气晴朗，温度为25°C今天北京的天气晴朗，温度为25°C。今天北京的天气晴朗，温度为25°C。
[消息完成] assistant

=== Agent 结束 (共 4 条消息) ===


[5] 演示steering功能...

=== Agent 启动 ===


--- 新回合开始 ---

[消息开始] user: ...

[消息完成] user

[消息开始] assistant: ...

[用户中断] 发送steering消息...
我来我来计算我来计算表达式我来计算表达式 `我来计算表达式 `

In [1]:
from nova_agent import AgentLoopConfig

In [3]:
print(AgentLoopConfig())

AgentLoopConfig(temperature=None, max_tokens=None, signal=None, api_key=None, transport=None, cache_retention=None, session_id=None, headers=None, on_payload=None, max_retry_delay_ms=None, metadata=None, reasoning=None, thinking_budgets=None, model=None, convert_to_llm=NotImplemented, transform_context=None, get_api_key=None, get_steering_messages=None, get_follow_up_messages=None)
